In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("../data/raw/application_train.csv")
print(df.shape)

(307511, 122)


In [8]:
print(df["TARGET"].value_counts(normalize=True))

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [9]:
faltantes = df.isna().mean().sort_values(ascending=False)
print(faltantes[faltantes > 0].head(10))

COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
dtype: float64


In [10]:
print((faltantes > 0.5).sum(), "colunas com mais de 50% de dados faltantes")

41 colunas com mais de 50% de dados faltantes


In [11]:
trios = [c for c in df.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]
print(len(trios), "colunas com sufixo de trio")
print(trios[:12])

47 colunas com sufixo de trio
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG']


In [12]:
from collections import defaultdict

grupos = defaultdict(list)
for c in trios:
    base, sufixo = c.rsplit("_", 1)
    grupos[base].append(sufixo)

incompletos = {b: s for b, s in grupos.items() if len(s) != 3}
print(len(grupos), "grupos")
print("grupos com menos de 3 versões:", incompletos)

19 grupos
grupos com menos de 3 versões: {'FONDKAPREMONT': ['MODE'], 'HOUSETYPE': ['MODE'], 'TOTALAREA': ['MODE'], 'WALLSMATERIAL': ['MODE'], 'EMERGENCYSTATE': ['MODE']}


In [13]:
print(df[trios].dtypes.value_counts())
print(
    df[
        [
            "FONDKAPREMONT_MODE",
            "HOUSETYPE_MODE",
            "TOTALAREA_MODE",
            "WALLSMATERIAL_MODE",
            "EMERGENCYSTATE_MODE",
        ]
    ].dtypes
)

float64    43
str         4
Name: count, dtype: int64
FONDKAPREMONT_MODE         str
HOUSETYPE_MODE             str
TOTALAREA_MODE         float64
WALLSMATERIAL_MODE         str
EMERGENCYSTATE_MODE        str
dtype: object


In [14]:
categoricas = ["FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"]
print(faltantes[categoricas])
print((faltantes[categoricas] > 0.5).sum(), "de 4 com mais de 50% faltante")

FONDKAPREMONT_MODE     0.683862
HOUSETYPE_MODE         0.501761
WALLSMATERIAL_MODE     0.508408
EMERGENCYSTATE_MODE    0.473983
dtype: float64
3 de 4 com mais de 50% faltante


In [15]:
for g in ["LANDAREA", "ELEVATORS"]:
    cols = [f"{g}_AVG", f"{g}_MODE", f"{g}_MEDI"]
    print(g)
    print(df[cols].corr().round(3))

LANDAREA
               LANDAREA_AVG  LANDAREA_MODE  LANDAREA_MEDI
LANDAREA_AVG          1.000          0.974          0.992
LANDAREA_MODE         0.974          1.000          0.981
LANDAREA_MEDI         0.992          0.981          1.000
ELEVATORS
                ELEVATORS_AVG  ELEVATORS_MODE  ELEVATORS_MEDI
ELEVATORS_AVG           1.000           0.979           0.996
ELEVATORS_MODE          0.979           1.000           0.983
ELEVATORS_MEDI          0.996           0.983           1.000


In [16]:
completos = [b for b, s in grupos.items() if len(s) == 3]
descartar = [f"{b}_{s}" for b in completos for s in ["AVG", "MODE"]]

mis = df.isna().mean()
df2 = df.drop(columns=descartar)
alta = [c for c in df2.columns if mis[c] > 0.5 and c not in ("TARGET", "SK_ID_CURR")]
df3 = df2.drop(columns=alta)

features = [c for c in df3.columns if c not in ("TARGET", "SK_ID_CURR")]
print(len(descartar), "colunas descartadas na consolidação")
print(len(alta), "colunas descartadas por mais de 50% faltante")
print(len(features), "features restantes")

28 colunas descartadas na consolidação
17 colunas descartadas por mais de 50% faltante
75 features restantes
